# A13 Report — Problem B: MoveNet Good vs Bad Exercise Classification

This notebook reports the Deep Learning work for **Problem B** using two approaches:

1. **Dense neural network**
2. **CNN-based neural network**

The task is binary sequence classification:

- `1 = good exercise`
- `0 = bad exercise`

The input data consists of MoveNet 2D pose coordinates. Each sequence is cut to the exercise segment, converted to a fixed length of 10 frames, normalized, optionally extended with engineered distance features, and passed into a neural network classifier.

The full experimental notebooks tested several variants and used maximum augmentation.  
This report notebook is designed to be runnable, so it only trains the champion Dense and CNN configurations using a smaller augmentation setup.

## Deep Learning steps followed

The notebook follows these Deep Learning steps:

1. Load and prepare MoveNet Good/Bad sequence data.
2. Convert MoveNet joints to the common 13-joint assignment format.
3. Cut each sequence to the exercise segment using available start/stop predictions.
4. Normalize coordinate features using `MinMaxScaler` fitted on the training fold only.
5. Convert every sequence to exactly 10 sampled frames.
6. Add engineered distance features.
7. Train and evaluate the Dense champion model.
8. Train and evaluate the CNN champion model.
9. Compare Dense and CNN using cross-validation test metrics.
10. Discuss practical usefulness in a real-world exercise-feedback setting.

### Import

In [ ]:
from pathlib import Path
import os
import random
import warnings
import copy
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupKFold
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except ImportError:
    HAS_STRATIFIED_GROUP_KFOLD = False

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Paths and report settings

The full notebooks used the Good/Bad MoveNet CSV files and A11-predicted start/stop boundaries.

This report notebook tries to find the same files. If no start/stop file is found, the notebook falls back to using the full sequence.

## Champion configurations

The full Dense notebook tested several Dense variants. The best Dense champion was:

- feature set: `coords_distances`
- hidden layers: `(24, 12)`
- dropout: `0.10`
- input dimension: `380`
- parameters: `9457`

The full CNN notebook tested several small CNN variants. The best CNN champion was:

- feature set: `coords_distances`
- Conv1D channels: `(8,)`
- kernel size: `3`
- stride: `1`
- dense layers: `(8,)`
- dropout: `0.20`
- parameters: `1017`

The CNN has about 10% of the Dense model parameters.

In [ ]:
# Main data path used in the A13 notebooks
MOVENET_GOOD_BAD_DIR = Path("../../../data/processed/Good_vs_Bad_movenet/csv_output")

# Possible start/stop boundary files from A11/A13
CUT_BOUND_CANDIDATES = [
    Path("reporting/results/good_bad_start_stop_predictions_from_mlflow.csv"),
    Path("../A11/Kim/reporting/results/good_bad_start_stop_predictions.csv"),
    Path("../../A11/Kim/reporting/results/a11_problem_b_all_boundary_table.csv"),
    Path("../../../data/processed/good_bad_start_stop_predictions.csv"),
]

output_dir = Path("reporting/results")
artifact_dir = Path("reporting/artifacts")
output_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 10
REPORT_N_SPLITS = 10
VALIDATION_SPLIT = 0.10

EPOCHS = 120
PATIENCE = 15
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
THRESHOLD = 0.50

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

print("MOVENET_GOOD_BAD_DIR:", MOVENET_GOOD_BAD_DIR)
print("Directory exists:", MOVENET_GOOD_BAD_DIR.exists())

In [1]:
DENSE_CHAMPION_CONFIG = {
    "name": "dense_coords_dist_24_12_drop010",
    "model_family": "Dense",
    "seq_len": SEQ_LEN,
    "feature_set": "coords_distances",
    "hidden_dims": (24, 12),
    "dropout": 0.10,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "threshold": THRESHOLD,
}

CNN_CHAMPION_CONFIG = {
    "name": "cnn_8_k3_dist_drop020",
    "model_family": "CNN",
    "seq_len": SEQ_LEN,
    "feature_set": "coords_distances",
    "conv_channels": (8,),
    "kernel_size": 3,
    "stride": 1,
    "dense_dims": (8,),
    "dropout": 0.20,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "threshold": THRESHOLD,
}

print("Dense champion config:")
print(json.dumps({k: str(v) for k, v in DENSE_CHAMPION_CONFIG.items()}, indent=2))

print("\nCNN champion config:")
print(json.dumps({k: str(v) for k, v in CNN_CHAMPION_CONFIG.items()}, indent=2))

NameError: name 'SEQ_LEN' is not defined

## MoveNet loading helpers

MoveNet uses its own joint names. The notebook maps them to the 13-joint assignment convention.

The model uses 13 joints with x/y coordinates:

$
13 \text{ joints} \times 2 \text{ dimensions} = 26 \text{ coordinate features per frame}
$

The champion feature set also adds selected distance features, giving 38 features per frame.

In [ ]:
FRAME_COL_CANDIDATES = [
    "frame_index",
    "frame",
    "frame_id",
    "FrameNo",
    "frame_no",
    "image_id",
]

MOVENET_TO_ASSIGNMENT = {
    "nose": "head",
    "left_shoulder": "left_shoulder",
    "left_elbow": "left_elbow",
    "right_shoulder": "right_shoulder",
    "right_elbow": "right_elbow",
    "left_wrist": "left_hand",
    "right_wrist": "right_hand",
    "left_hip": "left_hip",
    "right_hip": "right_hip",
    "left_knee": "left_knee",
    "right_knee": "right_knee",
    "left_ankle": "left_foot",
    "right_ankle": "right_foot",
}

JOINTS_13 = list(MOVENET_TO_ASSIGNMENT.values())

COORD_COLS = []
for joint in JOINTS_13:
    COORD_COLS.extend([f"{joint}_x", f"{joint}_y"])


def strip_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def find_frame_col(df):
    for col in FRAME_COL_CANDIDATES:
        if col in df.columns:
            return col
    raise ValueError(f"No frame column found. Available columns: {df.columns.tolist()}")


def load_movenet_as_assignment_df(csv_path):
    df = pd.read_csv(csv_path)
    df = strip_columns(df)

    frame_col = find_frame_col(df)

    out = pd.DataFrame()
    out["frame_index"] = pd.to_numeric(df[frame_col], errors="coerce").astype(int)

    if "timestamp_ms" in df.columns:
        out["timestamp_ms"] = df["timestamp_ms"]

    for movenet_joint, assignment_joint in MOVENET_TO_ASSIGNMENT.items():
        for suffix in ["x", "y", "score"]:
            src = f"{movenet_joint}_{suffix}"
            dst = f"{assignment_joint}_{suffix}"

            if src not in df.columns:
                raise ValueError(f"Missing expected MoveNet column: {src} in {csv_path}")

            out[dst] = pd.to_numeric(df[src], errors="coerce")

    return out.sort_values("frame_index").reset_index(drop=True)


def sequence_label_from_name(sequence_id):
    stem = Path(sequence_id).stem

    if stem == "A1" or stem.startswith("G"):
        return 1

    if stem.startswith("W"):
        return 0

    raise ValueError(f"Cannot infer good/bad label from sequence_id: {sequence_id}")


def stem_base(path):
    return Path(path).stem


print("Number of coordinate columns:", len(COORD_COLS))
print("Coordinate columns:", COORD_COLS[:6], "...")

## Start/stop boundary helpers

The full pipeline uses A11 start/stop predictions to cut each sequence to the actual exercise segment.

If a sequence has a boundary prediction, the notebook uses it.  
If not, it falls back to the full sequence.

In [ ]:
def find_existing_cut_bounds_path():
    for p in CUT_BOUND_CANDIDATES:
        if p.exists():
            return p
    return None


def load_cut_bounds_table(cut_bounds_csv):
    if cut_bounds_csv is None or not Path(cut_bounds_csv).exists():
        print("No cut bounds CSV found. Full-sequence fallback will be used.")
        return None

    bounds_df = pd.read_csv(cut_bounds_csv)
    bounds_df.columns = [str(c).strip() for c in bounds_df.columns]

    if "sequence_id" not in bounds_df.columns:
        print("Cut bounds file has no sequence_id column. Full-sequence fallback will be used.")
        return None

    if {"start_frame", "stop_frame"}.issubset(bounds_df.columns):
        start_col, stop_col = "start_frame", "stop_frame"
    elif {"pred_start", "pred_stop"}.issubset(bounds_df.columns):
        start_col, stop_col = "pred_start", "pred_stop"
    elif {"true_start", "true_stop"}.issubset(bounds_df.columns):
        start_col, stop_col = "true_start", "true_stop"
    else:
        print("Cut bounds file has no recognized start/stop columns. Full-sequence fallback will be used.")
        return None

    out = bounds_df[["sequence_id", start_col, stop_col]].copy()
    out = out.rename(columns={start_col: "start_frame", stop_col: "stop_frame"})
    out["sequence_id"] = out["sequence_id"].astype(str)
    out["start_frame"] = pd.to_numeric(out["start_frame"], errors="coerce")
    out["stop_frame"] = pd.to_numeric(out["stop_frame"], errors="coerce")
    out = out.dropna(subset=["start_frame", "stop_frame"])

    out["start_frame"] = out["start_frame"].astype(int)
    out["stop_frame"] = out["stop_frame"].astype(int)

    return out


def get_sequence_bounds(sequence_id, df, bounds_lookup):
    if bounds_lookup is not None and sequence_id in bounds_lookup:
        start_frame, stop_frame = bounds_lookup[sequence_id]
        return int(start_frame), int(stop_frame), "cut_bounds_csv"

    start_frame = int(df["frame_index"].min())
    stop_frame = int(df["frame_index"].max())
    return start_frame, stop_frame, "full_sequence_fallback"


def cut_sequence(df, start_frame, stop_frame):
    start_frame, stop_frame = sorted([int(start_frame), int(stop_frame)])

    out = df[
        (df["frame_index"] >= start_frame)
        & (df["frame_index"] <= stop_frame)
    ].copy()

    return out.sort_values("frame_index").reset_index(drop=True)


CUT_BOUNDS_CSV = find_existing_cut_bounds_path()
print("Using cut bounds:", CUT_BOUNDS_CSV)

## Load all MoveNet Good/Bad sequences

This cell creates two objects:

- `catalog_df`: one row per sequence with metadata
- `sequence_dfs`: dictionary containing the actual cut frame data

The sequence labels are inferred from filenames:

- `A1` or `G...` = good = 1
- `W...` = bad = 0

In [ ]:
def load_good_bad_sequences(movenet_dir, cut_bounds_csv, min_cut_frames=SEQ_LEN):
    if not movenet_dir.exists():
        raise FileNotFoundError(
            f"MoveNet directory not found: {movenet_dir}\n"
            "Update MOVENET_GOOD_BAD_DIR to the folder containing the Good/Bad MoveNet CSV files."
        )

    cut_bounds_df = load_cut_bounds_table(cut_bounds_csv)

    bounds_lookup = None
    if cut_bounds_df is not None:
        bounds_lookup = {
            row["sequence_id"]: (row["start_frame"], row["stop_frame"])
            for _, row in cut_bounds_df.iterrows()
        }

    rows = []
    sequence_dfs = {}

    csv_files = sorted(movenet_dir.glob("*.csv"))
    print("Found MoveNet CSV files:", len(csv_files))

    for csv_path in csv_files:
        sequence_id = stem_base(csv_path)

        try:
            label = sequence_label_from_name(sequence_id)
        except ValueError:
            continue

        try:
            raw_df = load_movenet_as_assignment_df(csv_path)
            start_frame, stop_frame, cut_source = get_sequence_bounds(
                sequence_id=sequence_id,
                df=raw_df,
                bounds_lookup=bounds_lookup,
            )

            cut_df = cut_sequence(raw_df, start_frame, stop_frame)

            if len(cut_df) < min_cut_frames:
                rows.append({
                    "sequence_id": sequence_id,
                    "path": str(csv_path),
                    "label": label,
                    "start_frame": start_frame,
                    "stop_frame": stop_frame,
                    "cut_source": cut_source,
                    "n_frames_raw": len(raw_df),
                    "n_frames_cut": len(cut_df),
                    "kept": False,
                    "reason": "too_short_after_cut",
                })
                continue

            sequence_dfs[sequence_id] = cut_df

            rows.append({
                "sequence_id": sequence_id,
                "path": str(csv_path),
                "label": label,
                "start_frame": start_frame,
                "stop_frame": stop_frame,
                "cut_source": cut_source,
                "n_frames_raw": len(raw_df),
                "n_frames_cut": len(cut_df),
                "kept": True,
                "reason": "ok",
            })

        except Exception as e:
            rows.append({
                "sequence_id": sequence_id,
                "path": str(csv_path),
                "label": label,
                "start_frame": np.nan,
                "stop_frame": np.nan,
                "cut_source": "error",
                "n_frames_raw": np.nan,
                "n_frames_cut": np.nan,
                "kept": False,
                "reason": str(e),
            })

    catalog_df = pd.DataFrame(rows)

    if catalog_df.empty:
        raise RuntimeError("No valid MoveNet Good/Bad sequences were loaded.")

    return catalog_df, sequence_dfs


catalog_df, sequence_dfs = load_good_bad_sequences(
    movenet_dir=MOVENET_GOOD_BAD_DIR,
    cut_bounds_csv=CUT_BOUNDS_CSV,
    min_cut_frames=SEQ_LEN,
)

kept_catalog_df = catalog_df[catalog_df["kept"]].copy().reset_index(drop=True)

sequence_ids_all = kept_catalog_df["sequence_id"].tolist()
labels_lookup = dict(zip(kept_catalog_df["sequence_id"], kept_catalog_df["label"]))

print("Kept sequences:", len(sequence_ids_all))
print("Dropped sequences:", int((~catalog_df["kept"]).sum()))

display(kept_catalog_df.head())

print("\nLabel distribution:")
display(kept_catalog_df["label"].value_counts().rename(index={1: "good", 0: "bad"}))

print("\nCut source distribution:")
display(kept_catalog_df["cut_source"].value_counts())

## Feature engineering

The raw coordinate input has 26 features per frame:

$
13 \times 2 = 26
$

The champion models use `coords_distances`, meaning they add selected distances between joints.

The final feature count is:

$
26 \text{ coordinates} + 12 \text{ distances} = 38 \text{ features per frame}
$

Since each sequence is sampled to 10 frames:

$
10 \times 38 = 380
$

For the Dense model, this becomes one vector of 380 values.  
For the CNN model, this becomes a matrix with shape `(38 features, 10 frames)`.

In [ ]:
SELECTED_DISTANCE_PAIRS = [
    ("left_shoulder", "right_shoulder"),
    ("left_hip", "right_hip"),
    ("left_shoulder", "left_hip"),
    ("right_shoulder", "right_hip"),
    ("left_shoulder", "left_elbow"),
    ("left_elbow", "left_hand"),
    ("right_shoulder", "right_elbow"),
    ("right_elbow", "right_hand"),
    ("left_hip", "left_knee"),
    ("left_knee", "left_foot"),
    ("right_hip", "right_knee"),
    ("right_knee", "right_foot"),
]


def add_selected_distances(df):
    out = df.copy()

    for a, b in SELECTED_DISTANCE_PAIRS:
        dx = out[f"{a}_x"] - out[f"{b}_x"]
        dy = out[f"{a}_y"] - out[f"{b}_y"]
        out[f"dist_{a}_{b}"] = np.sqrt(dx**2 + dy**2)

    return out


def get_feature_columns(feature_set):
    if feature_set == "coords":
        return list(COORD_COLS)

    if feature_set == "coords_distances":
        distance_cols = [
            f"dist_{a}_{b}"
            for a, b in SELECTED_DISTANCE_PAIRS
        ]
        return list(COORD_COLS) + distance_cols

    raise ValueError(f"Unknown feature_set: {feature_set}")


def apply_feature_engineering(norm_df, feature_set):
    out = norm_df.copy()

    if feature_set == "coords_distances":
        out = add_selected_distances(out)

    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.fillna(0.0)

    return out


print("Raw coordinate features per frame:", len(COORD_COLS))
print("Distance features per frame:", len(SELECTED_DISTANCE_PAIRS))
print("Champion features per frame:", len(get_feature_columns("coords_distances")))

## Normalization and fixed-length sampling

Normalization is done with `MinMaxScaler`.

Important:

- The scaler is fitted only on the training fold.
- The same scaler is then applied to training, validation, and test data.

This avoids data leakage.

Every sequence is sampled to exactly 10 frames because Dense and CNN models need fixed-size inputs.

In [ ]:
def sample_fixed_length(df, seq_len):
    if len(df) < seq_len:
        raise ValueError(f"Sequence has {len(df)} frames, need seq_len={seq_len}.")

    indices = np.round(np.linspace(0, len(df) - 1, seq_len)).astype(int)
    return df.iloc[indices].copy().reset_index(drop=True)


def fit_minmax_scaler_on_sequences(sequence_ids, sequence_dfs):
    frames = []

    for sid in sequence_ids:
        frames.append(sequence_dfs[sid][COORD_COLS])

    train_coord_df = pd.concat(frames, ignore_index=True)

    scaler = MinMaxScaler()
    scaler.fit(train_coord_df)

    return scaler


def normalize_sequence_df(df, scaler):
    out = df.copy()
    out[COORD_COLS] = scaler.transform(out[COORD_COLS])
    return out


def sequence_to_dense_vector(df, scaler, seq_len, feature_set):
    norm_df = normalize_sequence_df(df, scaler)
    sampled_df = sample_fixed_length(norm_df, seq_len)
    feat_df = apply_feature_engineering(sampled_df, feature_set)

    values = feat_df[get_feature_columns(feature_set)].to_numpy(dtype=np.float32)
    return values.reshape(-1)


def sequence_to_cnn_matrix(df, scaler, seq_len, feature_set):
    norm_df = normalize_sequence_df(df, scaler)
    sampled_df = sample_fixed_length(norm_df, seq_len)
    feat_df = apply_feature_engineering(sampled_df, feature_set)

    values = feat_df[get_feature_columns(feature_set)].to_numpy(dtype=np.float32)

    # CNN input is features/channels by time.
    return values.T


def build_xy_for_sequence_ids(sequence_ids, sequence_dfs, labels_lookup, scaler, seq_len, feature_set, model_family):
    X_rows = []
    y_rows = []
    ids = []

    for sid in sequence_ids:
        if model_family == "Dense":
            X_rows.append(sequence_to_dense_vector(sequence_dfs[sid], scaler, seq_len, feature_set))
        elif model_family == "CNN":
            X_rows.append(sequence_to_cnn_matrix(sequence_dfs[sid], scaler, seq_len, feature_set))
        else:
            raise ValueError(f"Unknown model_family: {model_family}")

        y_rows.append(labels_lookup[sid])
        ids.append(sid)

    return np.stack(X_rows).astype(np.float32), np.array(y_rows, dtype=np.float32), ids

## Small augmentation used in this report notebook

The full Dense and CNN notebooks used maximum cascade augmentation:

- original
- horizontal mirror
- x scaling
- y scaling
- small positive rotation
- small negative rotation

The full augmentation expanded the training data by 32×:

$
90 \text{ original training sequences} \times 32 = 2880 \text{ training samples}
$

To keep this report notebook runnable, we use a smaller augmentation setup:

- original
- mirrored
- rotated +5 degrees
- rotated -5 degrees

This gives 4× training data expansion instead of 32×.

Validation and test data are not augmented.

In [ ]:
def transform_coords_2d(norm_df, mirror_x=False, rotation_degrees=0.0):
    out = norm_df.copy()

    xy = out[COORD_COLS].to_numpy(dtype=np.float32).reshape(len(out), len(JOINTS_13), 2)

    # Transform around the normalized coordinate center.
    xy = xy - 0.5

    if mirror_x:
        xy[:, :, 0] = -xy[:, :, 0]

    if rotation_degrees != 0:
        theta = np.deg2rad(rotation_degrees)
        rotation_matrix = np.array([
            [np.cos(theta), -np.sin(theta)],
            [np.sin(theta),  np.cos(theta)],
        ], dtype=np.float32)

        xy = xy @ rotation_matrix.T

    xy = xy + 0.5
    xy = np.clip(xy, 0.0, 1.0)

    out[COORD_COLS] = xy.reshape(len(out), len(COORD_COLS))

    return out


def make_small_augmentation_versions(norm_df):
    return [
        ("original", norm_df),
        ("mirror_x", transform_coords_2d(norm_df, mirror_x=True)),
        ("rotate_plus_5", transform_coords_2d(norm_df, rotation_degrees=5.0)),
        ("rotate_minus_5", transform_coords_2d(norm_df, rotation_degrees=-5.0)),
    ]


def dense_vector_from_normalized_df(norm_df, seq_len, feature_set):
    sampled_df = sample_fixed_length(norm_df, seq_len)
    feat_df = apply_feature_engineering(sampled_df, feature_set)
    values = feat_df[get_feature_columns(feature_set)].to_numpy(dtype=np.float32)
    return values.reshape(-1)


def cnn_matrix_from_normalized_df(norm_df, seq_len, feature_set):
    sampled_df = sample_fixed_length(norm_df, seq_len)
    feat_df = apply_feature_engineering(sampled_df, feature_set)
    values = feat_df[get_feature_columns(feature_set)].to_numpy(dtype=np.float32)
    return values.T


def build_augmented_train_xy(
    sequence_ids,
    sequence_dfs,
    labels_lookup,
    scaler,
    seq_len,
    feature_set,
    model_family,
    use_small_augmentation=True,
):
    X_rows = []
    y_rows = []
    ids = []

    for sid in sequence_ids:
        norm_df = normalize_sequence_df(sequence_dfs[sid], scaler)

        if use_small_augmentation:
            versions = make_small_augmentation_versions(norm_df)
        else:
            versions = [("original", norm_df)]

        for aug_name, aug_df in versions:
            if model_family == "Dense":
                X_rows.append(dense_vector_from_normalized_df(aug_df, seq_len, feature_set))
            elif model_family == "CNN":
                X_rows.append(cnn_matrix_from_normalized_df(aug_df, seq_len, feature_set))
            else:
                raise ValueError(f"Unknown model_family: {model_family}")

            y_rows.append(labels_lookup[sid])
            ids.append(f"{sid}__{aug_name}")

    return np.stack(X_rows).astype(np.float32), np.array(y_rows, dtype=np.float32), ids


print("Small augmentation expansion factor:", len(make_small_augmentation_versions(kept_catalog_df.iloc[0:0])))

## Dataset and evaluation helpers

The models are trained with binary cross-entropy loss using logits.

The model output is a raw logit.  
A sigmoid converts it to probability of `good`.

Prediction rule:

$
p(good) \geq 0.5 \Rightarrow good
$

$
p(good) < 0.5 \Rightarrow bad
$

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_loader(X, y, batch_size, shuffle):
    dataset = SequenceDataset(X, y)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def evaluate_model(model, loader, threshold=THRESHOLD):
    model.eval()

    criterion = nn.BCEWithLogitsLoss()

    all_probs = []
    all_preds = []
    all_true = []

    total_loss = 0.0
    n_samples = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            probs = torch.sigmoid(logits).detach().cpu().numpy()
            preds = (probs >= threshold).astype(int)

            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())
            all_true.extend(y_batch.detach().cpu().numpy().astype(int).tolist())

            total_loss += loss.item() * len(X_batch)
            n_samples += len(X_batch)

    y_true = np.array(all_true)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)

    auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan

    return {
        "loss": total_loss / max(n_samples, 1),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": auc,
        "tp": int(((y_true == 1) & (y_pred == 1)).sum()),
        "fp": int(((y_true == 0) & (y_pred == 1)).sum()),
        "tn": int(((y_true == 0) & (y_pred == 0)).sum()),
        "fn": int(((y_true == 1) & (y_pred == 0)).sum()),
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

## Dense champion model

The Dense champion receives a flattened vector:

$
10 \text{ frames} \times 38 \text{ features} = 380 \text{ input values}
$

The champion architecture is:

- input: 380
- hidden layer 1: 24 neurons
- hidden layer 2: 12 neurons
- output: 1 logit

The Dense model has 9457 trainable parameters.

In [ ]:
class DenseGoodBadClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims=(24, 12), dropout=0.10):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)


# Parameter check for Dense champion
dense_input_dim = SEQ_LEN * len(get_feature_columns("coords_distances"))

dense_param_check_model = DenseGoodBadClassifier(
    input_dim=dense_input_dim,
    hidden_dims=DENSE_CHAMPION_CONFIG["hidden_dims"],
    dropout=DENSE_CHAMPION_CONFIG["dropout"],
)

print("Dense input dimension:", dense_input_dim)
print("Dense trainable parameters:", count_trainable_parameters(dense_param_check_model))

## CNN champion model

The CNN champion receives the sequence as a matrix:

$
38 \text{ features} \times 10 \text{ frames}
$

The CNN treats the features as channels and the frames as the temporal axis.

The champion architecture is:

- Conv1D with 8 channels
- kernel size 3
- BatchNorm
- ReLU
- Dropout
- Adaptive average pooling
- Linear output layer

The CNN champion has 945 trainable parameters, about 10% of the Dense model.

In [ ]:
class CNNGoodBadClassifier(nn.Module):
    def __init__(
        self,
        n_features,
        seq_len,
        conv_channels=(8,),
        kernel_size=3,
        stride=1,
        dense_dims=(),
        dropout=0.20,
    ):
        super().__init__()

        conv_layers = []
        in_ch = n_features

        for out_ch in conv_channels:
            padding = kernel_size // 2

            conv_layers.extend([
                nn.Conv1d(
                    in_channels=in_ch,
                    out_channels=out_ch,
                    kernel_size=kernel_size,
                    stride=stride,
                    padding=padding,
                ),
                nn.BatchNorm1d(out_ch),
                nn.ReLU(),
            ])

            if dropout > 0:
                conv_layers.append(nn.Dropout(dropout))

            in_ch = out_ch

        self.cnn = nn.Sequential(*conv_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)

        dense_layers = []
        prev_dim = in_ch

        for h in dense_dims:
            dense_layers.append(nn.Linear(prev_dim, h))
            dense_layers.append(nn.ReLU())

            if dropout > 0:
                dense_layers.append(nn.Dropout(dropout))

            prev_dim = h

        dense_layers.append(nn.Linear(prev_dim, 1))

        self.head = nn.Sequential(*dense_layers)

    def forward(self, x):
        x = self.cnn(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x).squeeze(-1)


cnn_n_features = len(get_feature_columns("coords_distances"))

cnn_param_check_model = CNNGoodBadClassifier(
    n_features=cnn_n_features,
    seq_len=CNN_CHAMPION_CONFIG["seq_len"],
    conv_channels=CNN_CHAMPION_CONFIG["conv_channels"],
    kernel_size=CNN_CHAMPION_CONFIG["kernel_size"],
    stride=CNN_CHAMPION_CONFIG["stride"],
    dense_dims=CNN_CHAMPION_CONFIG["dense_dims"],
    dropout=CNN_CHAMPION_CONFIG["dropout"],
)

dense_params = count_trainable_parameters(dense_param_check_model)
cnn_params = count_trainable_parameters(cnn_param_check_model)

print("CNN number of features/channels:", cnn_n_features)
print("CNN trainable parameters:", cnn_params)
print("Dense trainable parameters:", dense_params)
print("CNN parameter share of Dense:", cnn_params / dense_params)

## Training helper with early stopping

The models are trained with:

- `BCEWithLogitsLoss`
- Adam optimizer
- validation-loss based early stopping
- patience = 15

Early stopping means training stops when validation loss does not improve for several epochs.

In [ ]:
def train_one_model(model, train_loader, val_loader, learning_rate, epochs, patience):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = np.inf
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    patience_counter = 0

    history = []

    for epoch in range(1, epochs + 1):
        model.train()

        total_train_loss = 0.0
        n_train = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * len(X_batch)
            n_train += len(X_batch)

        train_loss = total_train_loss / max(n_train, 1)

        train_metrics = evaluate_model(model, train_loader)
        val_metrics = evaluate_model(model, val_loader)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_metrics["accuracy"],
            "train_f1": train_metrics["f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1": val_metrics["f1"],
        })

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_state)

    summary = {
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "epochs_ran": len(history),
    }

    return model, pd.DataFrame(history), summary

## Cross-validation setup

The report uses grouped cross-validation.

The original sequence ID is used as the group. This prevents the same original sequence from appearing in both training and test data.

This is especially important when augmentation is used.

In [ ]:
y_all = kept_catalog_df["label"].to_numpy()
groups_all = kept_catalog_df["sequence_id"].to_numpy()
X_dummy = np.zeros(len(kept_catalog_df))

if HAS_STRATIFIED_GROUP_KFOLD:
    cv_splitter = StratifiedGroupKFold(
        n_splits=REPORT_N_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED,
    )
    cv_splits = list(cv_splitter.split(X_dummy, y_all, groups_all))
    print("Using StratifiedGroupKFold.")
else:
    cv_splitter = GroupKFold(n_splits=REPORT_N_SPLITS)
    cv_splits = list(cv_splitter.split(X_dummy, y_all, groups_all))
    print("Using GroupKFold.")

print("Number of CV folds:", len(cv_splits))

## Cross-validation experiment function

This function trains either the Dense champion or the CNN champion using the same splits.

For each fold:

1. Split into train/validation/test.
2. Fit scaler on training only.
3. Apply small augmentation to training only.
4. Train the model.
5. Evaluate on train, validation, and test.
6. Store metrics.

In [ ]:
def make_group_train_val_split(train_val_ids, labels_lookup, validation_split, random_state):
    labels = np.array([labels_lookup[sid] for sid in train_val_ids])

    return train_test_split(
        train_val_ids,
        test_size=validation_split,
        random_state=random_state,
        stratify=labels if len(np.unique(labels)) == 2 else None,
    )


def create_model_from_config(config, X_train):
    if config["model_family"] == "Dense":
        input_dim = X_train.shape[1]

        model = DenseGoodBadClassifier(
            input_dim=input_dim,
            hidden_dims=config["hidden_dims"],
            dropout=config["dropout"],
        )

    elif config["model_family"] == "CNN":
        n_features = X_train.shape[1]

        model = CNNGoodBadClassifier(
            n_features=n_features,
            seq_len=config["seq_len"],
            conv_channels=config["conv_channels"],
            kernel_size=config["kernel_size"],
            stride=config["stride"],
            dense_dims=config["dense_dims"],
            dropout=config["dropout"],
        )

    else:
        raise ValueError(f"Unknown model family: {config['model_family']}")

    return model.to(DEVICE)


def run_cv_champion(config, use_small_augmentation=True):
    fold_rows = []
    history_rows = []

    model_family = config["model_family"]

    for fold_idx, (train_val_idx, test_idx) in enumerate(cv_splits, start=1):
        train_val_ids = kept_catalog_df.iloc[train_val_idx]["sequence_id"].tolist()
        test_ids = kept_catalog_df.iloc[test_idx]["sequence_id"].tolist()

        train_ids, val_ids = make_group_train_val_split(
            train_val_ids=train_val_ids,
            labels_lookup=labels_lookup,
            validation_split=VALIDATION_SPLIT,
            random_state=RANDOM_SEED + fold_idx,
        )

        scaler = fit_minmax_scaler_on_sequences(train_ids, sequence_dfs)

        X_train, y_train, _ = build_augmented_train_xy(
            train_ids,
            sequence_dfs,
            labels_lookup,
            scaler,
            config["seq_len"],
            config["feature_set"],
            model_family=model_family,
            use_small_augmentation=use_small_augmentation,
        )

        X_val, y_val, _ = build_xy_for_sequence_ids(
            val_ids,
            sequence_dfs,
            labels_lookup,
            scaler,
            config["seq_len"],
            config["feature_set"],
            model_family=model_family,
        )

        X_test, y_test, _ = build_xy_for_sequence_ids(
            test_ids,
            sequence_dfs,
            labels_lookup,
            scaler,
            config["seq_len"],
            config["feature_set"],
            model_family=model_family,
        )

        model = create_model_from_config(config, X_train)

        train_loader = make_loader(X_train, y_train, config["batch_size"], shuffle=True)
        val_loader = make_loader(X_val, y_val, config["batch_size"], shuffle=False)
        test_loader = make_loader(X_test, y_test, config["batch_size"], shuffle=False)

        model, history_df, train_summary = train_one_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            learning_rate=config["learning_rate"],
            epochs=config["epochs"],
            patience=config["patience"],
        )

        train_metrics = evaluate_model(model, train_loader, threshold=config["threshold"])
        val_metrics = evaluate_model(model, val_loader, threshold=config["threshold"])
        test_metrics = evaluate_model(model, test_loader, threshold=config["threshold"])

        n_parameters = count_trainable_parameters(model)

        fold_row = {
            "model_family": model_family,
            "model_name": config["name"],
            "fold": fold_idx,
            "feature_set": config["feature_set"],
            "n_parameters": n_parameters,
            "n_train_original_sequences": len(train_ids),
            "n_train_samples_after_aug": len(X_train),
            "n_val_sequences": len(val_ids),
            "n_test_sequences": len(test_ids),
            "best_val_loss": train_summary["best_val_loss"],
            "best_epoch": train_summary["best_epoch"],
            "epochs_ran": train_summary["epochs_ran"],
            "train_accuracy": train_metrics["accuracy"],
            "train_precision": train_metrics["precision"],
            "train_recall": train_metrics["recall"],
            "train_f1": train_metrics["f1"],
            "train_auc": train_metrics["auc"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_auc": val_metrics["auc"],
            "test_accuracy": test_metrics["accuracy"],
            "test_precision": test_metrics["precision"],
            "test_recall": test_metrics["recall"],
            "test_f1": test_metrics["f1"],
            "test_auc": test_metrics["auc"],
            "test_tp": test_metrics["tp"],
            "test_fp": test_metrics["fp"],
            "test_tn": test_metrics["tn"],
            "test_fn": test_metrics["fn"],
        }

        fold_rows.append(fold_row)

        history_df["model_family"] = model_family
        history_df["model_name"] = config["name"]
        history_df["fold"] = fold_idx
        history_rows.append(history_df)

        print(
            f"{model_family} | fold {fold_idx}/{REPORT_N_SPLITS} | "
            f"test acc={test_metrics['accuracy']:.3f} "
            f"precision={test_metrics['precision']:.3f} "
            f"recall={test_metrics['recall']:.3f} "
            f"f1={test_metrics['f1']:.3f} "
            f"params={n_parameters}"
        )

    fold_df = pd.DataFrame(fold_rows)
    history_all_df = pd.concat(history_rows, ignore_index=True)

    return fold_df, history_all_df

## Run Dense champion cross-validation

This report notebook trains only the Dense champion configuration.

The full Dense notebook used 32× augmentation.  
This report notebook uses 4× augmentation for runtime.

In [ ]:
dense_fold_df, dense_history_df = run_cv_champion(
    DENSE_CHAMPION_CONFIG,
    use_small_augmentation=True,
)

dense_summary_df = (
    dense_fold_df
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)

display(dense_fold_df)
display(dense_summary_df)

dense_fold_path = output_dir / "a13_report_dense_champion_folds.csv"
dense_summary_path = output_dir / "a13_report_dense_champion_summary.csv"

dense_fold_df.to_csv(dense_fold_path, index=False)
dense_summary_df.to_csv(dense_summary_path, index=False)

print("Saved:", dense_fold_path)
print("Saved:", dense_summary_path)

## Run CNN champion cross-validation

This report notebook trains only the CNN champion configuration.

The CNN champion has approximately 1017 parameters, compared with the Dense model's 9457 parameters.

In [ ]:
cnn_fold_df, cnn_history_df = run_cv_champion(
    CNN_CHAMPION_CONFIG,
    use_small_augmentation=True,
)

cnn_summary_df = (
    cnn_fold_df
    .select_dtypes(include=[np.number])
    .agg(["mean", "std"])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)

display(cnn_fold_df)
display(cnn_summary_df)

cnn_fold_path = output_dir / "a13_report_cnn_champion_folds.csv"
cnn_summary_path = output_dir / "a13_report_cnn_champion_summary.csv"

cnn_fold_df.to_csv(cnn_fold_path, index=False)
cnn_summary_df.to_csv(cnn_summary_path, index=False)

print("Saved:", cnn_fold_path)
print("Saved:", cnn_summary_path)

## Compare Dense and CNN report results

The most important metrics are the **test metrics**, because the assignment asks for evaluation on test data.

Training metrics are useful for checking overfitting.  
Validation metrics are useful for early stopping and model selection.  
Test metrics are used for the final comparison.

In [ ]:
def extract_mean_metric(summary_df, metric_name):
    row = summary_df[summary_df["metric"] == metric_name]
    if row.empty:
        return np.nan
    return float(row["mean"].iloc[0])


comparison_rows = []

for name, summary in [
    ("Dense report run", dense_summary_df),
    ("CNN report run", cnn_summary_df),
]:
    comparison_rows.append({
        "model": name,
        "parameters": extract_mean_metric(summary, "n_parameters"),
        "test_accuracy": extract_mean_metric(summary, "test_accuracy"),
        "test_precision": extract_mean_metric(summary, "test_precision"),
        "test_recall": extract_mean_metric(summary, "test_recall"),
        "test_f1": extract_mean_metric(summary, "test_f1"),
        "test_auc": extract_mean_metric(summary, "test_auc"),
        "test_fp_per_fold": extract_mean_metric(summary, "test_fp"),
        "test_fn_per_fold": extract_mean_metric(summary, "test_fn"),
    })

comparison_df = pd.DataFrame(comparison_rows)

dense_params_report = comparison_df.loc[comparison_df["model"] == "Dense report run", "parameters"].iloc[0]
comparison_df["parameter_share_vs_dense"] = comparison_df["parameters"] / dense_params_report

display(comparison_df)

## Full notebook reference results

The report run above is intentionally smaller because it uses reduced augmentation.

The full notebooks used maximum cascade augmentation, where each original training sequence becomes 32 training samples.

The full Dense champion achieved approximately (mean values):

- test accuracy: 0.9106
- test precision: 0.9085
- test recall: 0.9571
- test F1: 0.9312
- test AUC: 0.9714
- parameters: 9457

The full CNN champion achieved approximately (mean values):

- test accuracy: 0.9038
- test precision: 0.9052
- test recall: 0.9571
- test F1: 0.9248
- test AUC: 0.9686
- parameters: 1017

The CNN therefore performed better than the Dense model while using about 10% of the Dense model's parameters.

In [ ]:
full_results_df = pd.DataFrame([
    {
        "model": "Dense full notebook",
        "parameters": 9457,
        "test_accuracy": 0.910606,
        "test_precision": 0.908532,
        "test_recall": 0.957143,
        "test_f1": 0.931261,
        "test_auc": 0.971429,
        "test_fp_total_cv": 7,
        "test_fn_total_cv": 3,
    },
    {
        "model": "CNN full notebook",
        "parameters": 945,
        "test_accuracy": 0.972727,
        "test_precision": 0.987500,
        "test_recall": 0.971429,
        "test_f1": 0.977949,
        "test_auc": 0.996429,
        "test_fp_total_cv": 1,
        "test_fn_total_cv": 2,
    },
])

dense_params_full = full_results_df.loc[
    full_results_df["model"] == "Dense full notebook",
    "parameters"
].iloc[0]

full_results_df["parameter_share_vs_dense"] = full_results_df["parameters"] / dense_params_full

display(full_results_df)

metric_cols = ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_auc"]

improvement = {}
for col in metric_cols:
    dense_value = full_results_df.loc[full_results_df["model"] == "Dense full notebook", col].iloc[0]
    cnn_value = full_results_df.loc[full_results_df["model"] == "CNN full notebook", col].iloc[0]
    improvement[col] = cnn_value - dense_value

improvement_df = pd.DataFrame([
    {"metric": metric, "cnn_minus_dense": value}
    for metric, value in improvement.items()
])

display(improvement_df)

## Assignment requirement: CNN parameter budget

The assignment says that the smaller CNN models should have about 20% of the Dense model parameters and perform almost as well, with a maximum of 10% worse test metrics.

The Dense champion has 9457 parameters.

20% of the Dense model is:

$
9457 \times 0.20 = 1891
$

The CNN champion has 1017 parameters:

$
1017 / 9457 \approx 0.10
$

So the CNN is even smaller than the 20% target.

The CNN also performs better than the Dense model in the full notebook results, so it satisfies the requirement.

In [ ]:
dense_full_params = 9457
cnn_full_params = 945

parameter_budget_20_percent = dense_full_params * 0.20
cnn_parameter_share = cnn_full_params / dense_full_params

print("Dense parameters:", dense_full_params)
print("20% parameter budget:", parameter_budget_20_percent)
print("CNN parameters:", cnn_full_params)
print("CNN parameter share:", cnn_parameter_share)

dense_f1 = 0.931261
cnn_f1 = 0.977949
minimum_allowed_cnn_f1 = dense_f1 * 0.90

print("\nDense full test F1:", dense_f1)
print("Minimum allowed CNN F1 if max 10% worse:", minimum_allowed_cnn_f1)
print("CNN full test F1:", cnn_f1)
print("Requirement satisfied:", cnn_f1 >= minimum_allowed_cnn_f1)

## Dead ends and what did not work as well

### Dense model

The full Dense notebook tested several variants:

- coordinates only
- coordinates + distances
- coordinates + distances + angles
- different hidden layer sizes
- different dropout values
- lower learning rate

The best Dense result came from the smaller model using coordinates + distances:

- hidden layers: `(24, 12)`
- dropout: `0.10`
- feature set: `coords_distances`

The dead ends were:

1. **Coordinates only** did not work as well as coordinates + distances.
2. **Adding angle features** did not improve the champion result enough to justify the larger input.
3. **Larger Dense models** were not clearly better, likely because the dataset is small.
4. **Maximum augmentation** improved the Dense model only slightly.

### CNN model

The full CNN notebook tested several small CNN variants under the parameter budget.

The best CNN result came from:

- feature set: `coords_distances`
- one Conv1D layer with 8 channels
- kernel size 3
- no dense hidden head
- dropout 0.20
- 945 parameters

The dead ends were:

1. Smaller coordinate-only CNNs had fewer parameters but did not make full use of distance information.
2. More complex CNN variants were not necessary.
3. The best CNN was simple, small, and parameter-efficient.

Overall, the CNN architecture was better suited to the sequence format because it can learn local temporal patterns across the 10 sampled frames.

## Evaluation in the practical application context

The practical task is to decide whether an exercise sequence is good or bad.

The most important practical error is a **false positive**:

\[
\text{bad exercise predicted as good}
\]

This is problematic because a user could receive positive feedback even though the exercise form is bad.

In the full Dense notebook, the aggregated confusion matrix was approximately:

$
\begin{bmatrix}
34 & 7 \\
3 & 68
\end{bmatrix}
$

This means:

- 34 bad exercises were correctly classified as bad.
- 7 bad exercises were incorrectly classified as good.
- 3 good exercises were incorrectly classified as bad.
- 68 good exercises were correctly classified as good.

In the full CNN notebook, the aggregated confusion matrix was approximately:

$
\begin{bmatrix}
40 & 1 \\
2 & 69
\end{bmatrix}
$

This means the CNN reduced false positives from 7 to 1.

Therefore, the CNN is more useful in the practical application context because it is less likely to classify a bad exercise as good.

In [ ]:
dense_cm = np.array([
    [34, 7],
    [3, 68],
])

cnn_cm = np.array([
    [40, 1],
    [2, 69],
])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, cm, title in [
    (axes[0], dense_cm, "Dense full CV confusion matrix"),
    (axes[1], cnn_cm, "CNN full CV confusion matrix"),
]:
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Bad", "Good"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Bad", "Good"])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white", fontsize=14)

plt.tight_layout()
plt.show()

## Final conclusion

The Dense champion achieved strong performance with approximately 9457 parameters. It reached about 0.931 test F1 and 0.971 test AUC in the full notebook.

The CNN champion achieved even stronger performance while using only 945 parameters, around 10% of the Dense model size. It reached about 0.978 test F1 and 0.996 test AUC.

This satisfies the assignment requirement because the CNN is much smaller than the Dense model and is not worse by 10%; it is actually better on the test metrics.

From a practical perspective, the CNN is also more useful because it reduces false positives. A false positive means a bad exercise is classified as good, which is the more serious error in an exercise-feedback system.

Overall, the best setup for Problem B is the CNN champion:

- input: MoveNet 2D
- sequence length: 10 frames
- feature set: coordinates + selected distances
- Conv1D channels: 8
- kernel size: 3
- dropout: 0.20
- parameters: 945
- test F1: approximately 0.978